# Tome S2, Chapter 2: Transformations (Python companion)

Exercises first, solutions at the end. We classify isometries from their formulas, draw and recognise friezes, hunt for integer matrices of finite order (the crystallographic restriction), compute point groups of lattices, build the rotation group of the cube, and play billiards in a square. We use `numpy`, `matplotlib` and the standard library.

*Before each solution, predict what the output will look like.*

In [1]:
import cmath, math, itertools
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2)

## Exercises

**P2.1 (★) Naming an isometry.** Write a function that takes an isometry $z\mapsto e^{i\theta}z+b$ or $z\mapsto e^{i\theta}\bar z+b$ (as in Theorem 2.3) and returns its kind together with its centre, or its axis and glide vector. Test it on products of two and three random reflections.

**P2.2 (★★) Recognising friezes.** Generate the seven frieze patterns of Theorem 2.7 as finite point sets (an asymmetric motif and its images), then write a recogniser that tests which of the four kinds of symmetry (vertical mirror, horizontal mirror, half-turn, glide) map the pattern to itself, and check that it recovers the seven names.

**P2.3 (★) Integer matrices of finite order.** Find all $2\times2$ integer matrices with entries in $\{-3,\dots,3\}$ that have finite order, and list the orders that occur. Do the same for $3\times3$ matrices with entries in $\{-1,0,1\}$. Compare with Theorem 2.8.

**P2.4 (★★) Point groups of lattices.** Compute the order of the point group of the lattices of Figure 2.3 and of a random lattice, by testing the rotations and reflections that permute the short vectors (Proposition 2.9).

**P2.5 (★) The cube.** Generate the rotation group of the cube from the rotation by $90^\circ$ about a face axis and the rotation by $120^\circ$ about a diagonal, and count its elements by order (Exercise 2.9).

**P2.6 (★) Billiards.** Simulate a billiard ball in the unit square with initial slope $p/q$ and measure the length of the path until it returns to its starting point with its starting direction (Exercise 2.10).

## Solutions

### P2.1
**How to find it.** Follow the proof of Theorem 2.3: the fixed point of a direct map is $b/(1-e^{i\theta})$; an indirect map conjugated by the rotation by $\theta/2$ becomes $\bar z+b'$.

In [2]:
def name(theta, b, direct, eps=1e-9):
    u = cmath.exp(1j * theta)
    if direct:
        if abs(u - 1) < eps:
            return ("identity",) if abs(b) < eps else ("translation", b)
        return ("rotation", b / (1 - u), theta % (2 * math.pi))
    v = cmath.exp(1j * theta / 2)
    bp = b / v                        # the map is conjugate to z -> conj(z) + bp
    t, s = bp.real, bp.imag           # reflection in Im z = s/2, then translation by t
    point = v * (1j * s / 2)          # a point of the axis, rotated back
    if abs(t) < eps:
        return ("reflection", "axis through", point, "direction", v)
    return ("glide", "axis through", point, "direction", v, "glide vector", t * v)

def reflection(p, d):                  # reflection in the line through p with unit direction d
    u = d * d                          # z -> u conj(z - p) + p = u conj(z) + (p - u conj(p))
    return (cmath.phase(u), p - u * p.conjugate(), False)

def compose(f, g):                     # f after g, each (theta, b, direct)
    tf, bf, df = f; tg, bg, dg = g
    uf, ug = cmath.exp(1j * tf), cmath.exp(1j * tg)
    if df:   # f(z) = uf z + bf
        return (tf + tg, uf * bg + bf, dg)
    # f(z) = uf conj(z) + bf
    return (tf - tg, uf * bg.conjugate() + bf, not dg)

def rand_reflection():
    p = complex(*rng.standard_normal(2)); d = cmath.exp(1j * rng.uniform(0, math.pi))
    return reflection(p, d)

f2 = compose(rand_reflection(), rand_reflection())
f3 = compose(rand_reflection(), compose(rand_reflection(), rand_reflection()))
print(name(*f2)[0], "  ", name(*f3)[0])
print(name(*compose(reflection(0, 1), reflection(1j, 1))))   # mirror Im z = 1 first, then the real axis

rotation    glide
('translation', -2j)


Two random reflections give a rotation (parallel mirrors have probability zero), three give a glide reflection (a pure reflection also has probability zero). Reflecting first in $\operatorname{Im}z=1$ and then in the real axis gives the translation by $-2i$, twice the vector from the first mirror to the second, as in Exercise 2.1.

### P2.2
**How to find it.** A finite piece of a frieze is only invariant up to its ends, so we test each candidate symmetry on the middle part of the pattern: every image of a middle point must be (numerically) a point of the pattern.

In [3]:
motif = np.array([0.10 + 0.08j, 0.36 + 0.08j, 0.36 + 0.17j, 0.19 + 0.17j, 0.19 + 0.42j, 0.10 + 0.42j, 0.25 + 0.12j])
I = lambda z: z
vert, horiz, half = (lambda z: -np.conj(z)), np.conj, (lambda z: -z)
glide = lambda z: np.conj(z) + 0.5
groups = {"hop": [I], "step": [I, glide], "sidle": [I, vert], "spinning hop": [I, half],
          "spinning sidle": [I, vert, glide, lambda z: vert(glide(z))], "jump": [I, horiz],
          "spinning jump": [I, vert, horiz, half]}

def pattern(reps, K=12):
    return np.concatenate([g(motif) + k for k in range(-K, K + 1) for g in reps])

def symmetries(P):
    mid = P[np.abs(P.real) < 4]
    def ok(f):
        Q = f(mid)
        return all(np.min(np.abs(P - q)) < 1e-9 for q in Q)
    found = set()
    for c in np.arange(0, 1, 0.25):               # candidate centres and mirror positions
        if ok(lambda z: -np.conj(z) + 2 * c): found.add("vertical mirror")
        if ok(lambda z: -z + 2 * c): found.add("half-turn")
        if ok(lambda z: np.conj(z) + c):
            found.add("horizontal mirror" if c == 0 else "glide")
    return found

def recognise(found):
    v, h, r, g = ("vertical mirror" in found, "horizontal mirror" in found, "half-turn" in found, "glide" in found)
    if h: return "spinning jump" if v else "jump"
    if v: return "spinning sidle" if (g or r) else "sidle"
    if g: return "step"
    return "spinning hop" if r else "hop"

for nm, reps in groups.items():
    print(f"{nm:15s} -> {recognise(symmetries(pattern(reps)))}")

hop             -> hop
step            -> step
sidle           -> sidle
spinning hop    -> spinning hop
spinning sidle  -> spinning sidle
jump            -> jump
spinning jump   -> spinning jump


The recogniser uses the logic of the proof: the horizontal mirror and the vertical mirrors decide most cases, and among groups without the horizontal mirror a glide or a half-turn separates the spinning sidle from the sidle and the step from the hop. Note that the jump also contains glides (the horizontal mirror followed by a translation), which is why the test for the horizontal mirror comes first.

### P2.3
**How to find it.** A matrix of finite order $n$ satisfies $A^n=I$; the crystallographic restriction says $n\le6$ in dimensions $2$ and $3$, but to be honest we test powers up to $24$.

In [4]:
def order(A, maxk=24):
    P = np.eye(len(A), dtype=np.int64)
    for k in range(1, maxk + 1):
        P = P @ A
        if np.abs(P).max() > 10**6:
            return None
        if (P == np.eye(len(A), dtype=np.int64)).all():
            return k
    return None

orders2 = {}
for e in itertools.product(range(-3, 4), repeat=4):
    A = np.array(e, dtype=np.int64).reshape(2, 2)
    k = order(A)
    if k: orders2[k] = orders2.get(k, 0) + 1
print("2x2, entries in [-3,3]: number of matrices of each finite order", dict(sorted(orders2.items())))

orders3 = {}
for e in itertools.product((-1, 0, 1), repeat=9):
    A = np.array(e, dtype=np.int64).reshape(3, 3)
    if round(abs(np.linalg.det(A))) != 1: continue
    k = order(A)
    if k: orders3[k] = orders3.get(k, 0) + 1
print("3x3, entries in {-1,0,1}:", dict(sorted(orders3.items())))

2x2, entries in [-3,3]: number of matrices of each finite order {1: 1, 2: 37, 3: 12, 4: 10, 6: 12}


3x3, entries in {-1,0,1}: {1: 1, 2: 163, 3: 308, 4: 348, 6: 764}


Only the orders $1,2,3,4,6$ occur. (These matrices need not be orthogonal; but a finite group of integer matrices preserves the lattice $\Z^n$ and some inner product, the average of the standard one over the group, so the theorem applies to them too.)

### P2.4
**How to find it.** Collect the lattice vectors up to some length; a candidate isometry is a symmetry when it maps this finite set onto itself. Rotations by multiples of $30^\circ$ and reflections in the lines through $0$ at multiples of $15^\circ$ suffice for these lattices, whose mirrors (if any) are horizontal, vertical or at multiples of $30^\circ$ or $45^\circ$.

In [5]:
def short_vectors(a, b, r):
    pts = [m * a + n * b for m in range(-8, 9) for n in range(-8, 9)]
    return np.array([z for z in pts if 0 < abs(z) <= r + 1e-9])

def point_group_order(a, b):
    r = 1.05 * max(abs(a), abs(b), abs(a + b), abs(a - b))
    S = short_vectors(a, b, r)
    def ok(f):
        return all(np.min(np.abs(S - f(z))) < 1e-9 for z in S)
    cands = [lambda z, k=k: z * cmath.exp(1j * k * math.pi / 6) for k in range(12)]
    cands += [lambda z, t=t: cmath.exp(2j * t) * np.conj(z) for t in (k * math.pi / 12 for k in range(12))]
    return sum(ok(f) for f in cands)

w = cmath.exp(1j * math.pi / 3)
for nm, a, b in [("oblique", 1, 0.35 + 0.8j), ("rectangular", 1, 0.75j), ("centred rectangular", 1.2, 0.6 + 0.45j),
                 ("square", 1, 1j), ("hexagonal", 1, w), ("random", complex(*rng.standard_normal(2)), complex(*rng.standard_normal(2)))]:
    print(f"{nm:20s} point group of order {point_group_order(a, b)}")

oblique              point group of order 2
rectangular          point group of order 4
centred rectangular  point group of order 4
square               point group of order 8
hexagonal            point group of order 12
random               point group of order 2


The orders $2,4,4,8,12$ are those of $C_2$, $D_2$, $D_2$, $D_4$, $D_6$, and a random lattice has only $\pm1$. The orders do not distinguish the rectangular lattice from the centred one: Proposition 2.9 needs the finer information of how the lattice sits with respect to the mirrors.

### P2.5
**How to find it.** Close the set of generators under multiplication; matrices with entries in $\{-1,0,1\}$ can be stored as tuples.

In [6]:
r = np.array([[0, -1, 0], [1, 0, 0], [0, 0, 1]])        # 90 degrees about the z-axis
d = np.array([[0, 0, 1], [1, 0, 0], [0, 1, 0]])         # 120 degrees about the diagonal (1,1,1)
G = {tuple(np.eye(3, dtype=int).ravel())}
frontier = list(G)
while frontier:
    new = []
    for g in frontier:
        for h in (r, d):
            p = tuple((np.array(g).reshape(3, 3) @ h).ravel())
            if p not in G:
                G.add(p); new.append(p)
    frontier = new
by_order = {}
for g in G:
    by_order[order(np.array(g).reshape(3, 3))] = by_order.get(order(np.array(g).reshape(3, 3)), 0) + 1
print(len(G), "rotations; number of each order:", dict(sorted(by_order.items())))

24 rotations; number of each order: {1: 1, 2: 9, 3: 8, 4: 6}


$24$ rotations: $1$ identity, $9$ of order $2$ ($3$ half-turns about face axes and $6$ about edge axes), $8$ of order $3$ (diagonals) and $6$ of order $4$ (quarter-turns about face axes), matching the solution of Exercise 2.9.

### P2.6
**How to find it.** Move the ball from wall to wall, reflecting the direction; stop when position and direction are both back to their initial values.

In [7]:
def billiard_period(p, q, x0=(0.123, 0.456), max_bounces=10000):
    pos = np.array(x0, float); d = np.array([q, p], float) / math.hypot(p, q)
    start, d0, length = pos.copy(), d.copy(), 0.0
    for _ in range(max_bounces):
        ts = [((1 if d[i] > 0 else 0) - pos[i]) / d[i] if d[i] != 0 else np.inf for i in range(2)]
        i = int(np.argmin(ts)); t = ts[i]
        # does the straight path pass through the start before the next wall?
        s = start - pos
        tt = s @ d
        if 1e-9 < tt <= t and np.linalg.norm(s - tt * d) < 1e-9 and np.allclose(d, d0):
            return length + tt
        pos = pos + t * d; length += t
        d[i] = -d[i]
    return None

for p, q in [(1, 1), (1, 2), (2, 3), (3, 5)]:
    print(f"slope {p}/{q}: period {billiard_period(p, q):.6f}, 2 sqrt(p^2+q^2) = {2 * math.hypot(p, q):.6f}")

slope 1/1: period 2.828427, 2 sqrt(p^2+q^2) = 2.828427
slope 1/2: period 4.472136, 2 sqrt(p^2+q^2) = 4.472136
slope 2/3: period 7.211103, 2 sqrt(p^2+q^2) = 7.211103
slope 3/5: period 11.661904, 2 sqrt(p^2+q^2) = 11.661904
